# NMODL 到 BrainCell Walkthrough

这个 notebook 按当前实现展示一条可检查的转换链：

```text
.mod
-> parser AST
-> raw_blocks
-> canonical_blocks
-> bc_ast
-> semantic_ir
-> target_ir
-> rendered Python
-> render validation
```

这里不会再把流程描述成“CanonicalBlocks 直接进模板”。当前真正有用的分层是：

- Step 1: `raw_blocks` / `canonical_blocks` / `bc_ast`
- Step 2: `semantic_ir` / `target_ir`
- Step 3: rendered Python / validation

本 notebook 用三个输入文件说明当前行为：

- `kv.mod`: 单离子 `inf/tau` 正例
- `na_alpha_beta.mod`: 单离子 `alpha/beta` 正例
- `hh.mod`: 语义层能构建，但 target lowering 会拒绝的反例


In [ ]:
from pathlib import Path
import json
import sys

repo_root = Path.cwd()
if not (repo_root / "steps").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from steps import ast_to_json
from steps import resolve_mod_file
from steps import save_pipeline_artifacts
from steps import run_pipeline_from_path
from steps.inspect_ast import run as run_inspect_ast
from steps.inspect_ir import run as run_inspect_ir


def preview_json(value, limit=2000):
    text = json.dumps(value, indent=2, ensure_ascii=False)
    print(text[:limit])
    if len(text) > limit:
        print(f"\n... truncated, total_chars={len(text)}")


def summarize_step1(step1_result):
    return {
        "source_file": step1_result["source_file"],
        "ast_root_type": step1_result["ast_root_type"],
        "block_counts": step1_result["block_counts"],
        "nonempty_raw_blocks": [name for name, values in step1_result["raw_blocks"].items() if values],
        "bc_ast_keys": sorted(step1_result["bc_ast"].keys()),
    }


def summarize_step2(step2_result):
    return {
        "summary": step2_result["summary"],
        "semantic_ir_keys": sorted(step2_result["semantic_ir"].keys()),
        "target_ir_keys": sorted(step2_result["target_ir"].keys()),
    }


kv_mod = resolve_mod_file(str(repo_root / "mod_files" / "kv.mod"))
na_mod = resolve_mod_file(str(repo_root / "mod_files" / "na_alpha_beta.mod"))
hh_mod = resolve_mod_file(str(repo_root / "mod_files" / "hh.mod"))

step1_kv = run_inspect_ast(kv_mod)
step2_kv = run_inspect_ir(step1_kv)
pipeline_kv = run_pipeline_from_path(str(kv_mod))

print(kv_mod)


## 1. 从输入文件开始

先看这次 walkthrough 的主正例 `kv.mod`。它是当前最适合观察完整链路的机制：

- 单 `USEION`
- one ohmic current
- `inf/tau` 门控动力学


In [ ]:
print(step1_kv["reconstructed_nmodl"])


## 2. parser AST 摘要

原始 parser AST 仍然是最底层真相来源，但 notebook 不再把它当作唯一可用接口。
这里先看根节点类型和 JSON 摘要。


In [ ]:
ast_payload_kv = step1_kv["ast_json"]
print(step1_kv["ast_root_type"])
preview_json(ast_payload_kv, limit=1500)


## 3. Step 1: raw_blocks / canonical_blocks / bc_ast

当前 Step 1 不只是“抽 block 文本”，而是同时产出三种边界：

- `raw_blocks`: 尽量保留原始块文本
- `canonical_blocks`: 面向块级规范化
- `bc_ast`: BrainCell 自己的 typed AST


In [ ]:
preview_json(summarize_step1(step1_kv), limit=2000)


In [ ]:
print("NEURON canonical block:")
preview_json(step1_kv["canonical_blocks"]["NEURON"], limit=1200)

print("\nBREAKPOINT canonical block:")
preview_json(step1_kv["canonical_blocks"]["BREAKPOINT"], limit=1200)

print("\nbc_ast summary:")
preview_json({
    "mechanism_name": step1_kv["bc_ast"]["mechanism_name"],
    "useions": step1_kv["bc_ast"]["useions"],
    "states": step1_kv["bc_ast"]["states"],
    "solve_blocks": step1_kv["bc_ast"]["solve_blocks"],
}, limit=2000)


## 4. Step 2: semantic_ir

`semantic_ir` 是当前实现最重要的新边界。它负责恢复：

- 符号表视角的 `symbols`
- 过程展开后的 `procedure_env`
- `BREAKPOINT` 中的电流表达式
- 状态方程识别后的 `gate_kinetics`
- `unsupported_features`


In [ ]:
preview_json({
    "summary": step2_kv["summary"],
    "procedure_env": step2_kv["semantic_ir"]["procedure_env"],
    "currents": step2_kv["semantic_ir"]["currents"],
    "gate_kinetics": step2_kv["semantic_ir"]["gate_kinetics"],
    "unsupported_features": step2_kv["semantic_ir"]["unsupported_features"],
}, limit=3500)


## 5. Step 2: target_ir

`target_ir` 已经是 BrainCell renderer 可直接消费的目标层。
对当前 density-channel 路径来说，重点关注：

- `target_family`
- `base_class_name`
- `g_max_param`
- `gates`
- `current_model`
- `supported` / `rejection_reasons`


In [ ]:
preview_json({
    "summary": step2_kv["summary"],
    "g_max_param": step2_kv["target_ir"]["g_max_param"],
    "gates": step2_kv["target_ir"]["gates"],
    "current_model": step2_kv["target_ir"]["current_model"],
}, limit=4000)


## 6. Step 3: rendered Python 与 validation

当前 Step 3 不只负责渲染，它还会做一次真实的生成后验证：

- `compile`
- `exec`
- 取到目标 class

这比旧版“模板文本能打印出来”更接近实际可用性。


In [ ]:
step3_kv = pipeline_kv["step3_result"]
preview_json(step3_kv["summary"], limit=2000)
print("\nRendered preview:\n")
print("\n".join(step3_kv["rendered_text"].splitlines()[:80]))


## 7. 保存 walkthrough 产物

下面这一步会把当前正例的核心产物保存到 `examples/artifacts/kv/`：

- `ast.json`
- `raw_blocks.json`
- `canonical_blocks.json`
- `bc_ast.json`
- `semantic_ir.json`
- `target_ir.json`
- `rendered_channel.py`
- `validation.json`
- `metadata.json`


In [ ]:
artifact_dir = repo_root / "examples" / "artifacts" / kv_mod.stem
artifact_summary = save_pipeline_artifacts(pipeline_kv, artifact_dir)
preview_json(artifact_summary, limit=2000)


## 8. 第二个正例：`alpha/beta`

`na_alpha_beta.mod` 会走另一条目标家族：

- `target_family = hh_ohmic_alpha_beta`
- 在 semantic 层保留 `alpha_expression` / `beta_expression`
- 在 target 层派生出 `inf/tau` 视图供渲染使用


In [ ]:
step1_na = run_inspect_ast(na_mod)
step2_na = run_inspect_ir(step1_na)
pipeline_na = run_pipeline_from_path(str(na_mod))

preview_json({
    "summary": step2_na["summary"],
    "gate_kinetics": step2_na["semantic_ir"]["gate_kinetics"],
    "target_family": step2_na["target_ir"]["target_family"],
    "validation": pipeline_na["step3_result"]["validation"],
}, limit=3500)


## 9. 反例：多离子机制为什么在当前版本被拒绝

`hh.mod` 说明当前拒绝并不是 parser 失败，也不是 semantic recovery 失败。
实际情况是：

- `semantic_ir` 仍然能够构建
- 但 `target_ir` 会因为 `multi_useion` 和 `nonspecific_current` 被拒绝
- 因此不应该继续进入当前 renderer


In [ ]:
step1_hh = run_inspect_ast(hh_mod)
step2_hh = run_inspect_ir(step1_hh)

preview_json({
    "summary": step2_hh["summary"],
    "semantic_useions": step2_hh["semantic_ir"]["useions"],
    "semantic_unsupported_features": step2_hh["semantic_ir"]["unsupported_features"],
    "target_supported": step2_hh["target_ir"]["supported"],
    "target_rejection_reasons": step2_hh["target_ir"]["rejection_reasons"],
}, limit=3000)


## 10. 对应 CLI

如果 notebook 中的分层观察已经足够明确，对应 CLI 如下：

```bash
python examples/convert_mod/nmodl/steps/inspect_ast/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/steps/inspect_ir/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/steps/render/cli.py   examples/convert_mod/nmodl/mod_files/kv.mod

python examples/convert_mod/nmodl/examples/generate_braincell.py   examples/convert_mod/nmodl/mod_files/kv.mod
```

建议顺序仍然是：先看 `bc_ast`，再看 `semantic_ir`，再看 `target_ir`，最后才关心渲染和 validation。
